## Q2. 

Download a YouTube spam collection dataset available from this link (Links to an external site.).

This is a public set of comments collected for spam research. It has five datasets composed by 1,956 real messages extracted from five videos. These 5 videos are popular pop songs that were among the 10 most viewed on the collection period.
All the five dataset has the following attributes:

COMMENT_ID: Unique id representing the comment  
AUTHOR: Author id,  
DATE: Date the comment is posted,  
CONTENT: The comment,  
TAG: For spam 1, otherwise 0

For this exercise use any 4 of these 5 datasets to build a spam filter with Naive Bayes approach and use that filter to check the accuracy on the remaining dataset. Make sure to report the details of your training and the model. [6 points]

### import packages

In [1]:
import os, glob

import numpy as np
import pandas as pd

import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

### read data

#### For this exercise use any four of these five datasets to build a spam filter with the Naïve Bayes approach.

In [2]:
train_df = pd.DataFrame()

#Read data
files = glob.glob("./YouTube-Spam-Collection-v1/*")
files = sorted(files)[:-1]
for f in files:
    temp = pd.read_csv(f)
    print("(file, shape):", f, temp.shape)
    train_df = train_df.append(temp, ignore_index=True)
print("the shape of training files", train_df.shape)
#Pick necessary columns
train_df = train_df[['CONTENT', 'CLASS']]
#train_df['CONTENT'] = train_df['CONTENT'].apply(textcleaner)
train_df.head(50)

(file, shape): ./YouTube-Spam-Collection-v1/Youtube01-Psy.csv (350, 5)
(file, shape): ./YouTube-Spam-Collection-v1/Youtube02-KatyPerry.csv (350, 5)
(file, shape): ./YouTube-Spam-Collection-v1/Youtube03-LMFAO.csv (438, 5)
(file, shape): ./YouTube-Spam-Collection-v1/Youtube04-Eminem.csv (448, 5)
the shape of training files (1586, 5)


/var/folders/nl/0t1qmzl174xchcymt5p1_p0w0000gn/T/ipykernel_2084/1049936703.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  train_df = train_df.append(temp, ignore_index=True)
/var/folders/nl/0t1qmzl174xchcymt5p1_p0w0000gn/T/ipykernel_2084/1049936703.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  train_df = train_df.append(temp, ignore_index=True)
/var/folders/nl/0t1qmzl174xchcymt5p1_p0w0000gn/T/ipykernel_2084/1049936703.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  train_df = train_df.append(temp, ignore_index=True)
/var/folders/nl/0t1qmzl174xchcymt5p1_p0w0000gn/T/ipykernel_2084/1049936703.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use panda

,CONTENT,CLASS
0,"Huh, anyway check out this you[tube] channel: ...",1
1,Hey guys check out my new channel and our firs...,1
2,just for test I have to say murdev.com,1
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1
4,watch?v=vtaRGgvGtWQ Check this out .﻿,1
5,"Hey, check out my new website!! This site is a...",1
6,Subscribe to my channel ﻿,1
7,i turned it on mute as soon is i came on i jus...,0
8,You should check my channel for Funny VIDEOS!!﻿,1
9,and u should.d check my channel and tell me wh...,1


In [3]:
test_df = pd.read_csv('./YouTube-Spam-Collection-v1/Youtube05-Shakira.csv')
print("the shape of the test dataset", test_df.shape)
test_df = test_df[['CONTENT', 'CLASS']]
#test_df['CONTENT'] = test_df['CONTENT'].apply(textcleaner)
test_df.head(50)

the shape of the test dataset (370, 5)


,CONTENT,CLASS
0,Nice song﻿,0
1,I love song ﻿,0
2,I love song ﻿,0
3,"860,000,000 lets make it first female to reach...",0
4,shakira is best for worldcup﻿,0
5,The best world cup song ever!!!!﻿,0
6,I love﻿,0
7,SEE SOME MORE SONG OPEN GOOGLE AND TYPE Shakir...,1
8,Awesome ﻿,0
9,I like shakira..﻿,0


In [4]:
def textcleaner(row):
    row = row.lower()     
    row = re.sub("[^\w\s']", "", row) # remove punctuations
    row = row.strip(" ")
    return row

In [5]:
test_df = pd.read_csv('./YouTube-Spam-Collection-v1/Youtube05-Shakira.csv')
print("the shape of the test dataset", test_df.shape)
test_df = test_df[['CONTENT', 'CLASS']]
test_df['CONTENT'] = test_df['CONTENT'].apply(textcleaner)
test_df.head(50)

the shape of the test dataset (370, 5)


,CONTENT,CLASS
0,nice song,0
1,i love song,0
2,i love song,0
3,860000000 lets make it first female to reach o...,0
4,shakira is best for worldcup,0
5,the best world cup song ever,0
6,i love,0
7,see some more song open google and type shakir...,1
8,awesome,0
9,i like shakira,0


In [6]:
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(train_df['CONTENT'])
vectorizer.get_feature_names_out()

array(['00', '000', '002', ..., 'ｆａｎｃy', 'ｉｓ', 'ｔｈｉｓ'], dtype=object)

In [7]:
print(X.toarray())

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


### apply Naive Bayes approach

In [10]:
#Create a data pipeline
pipeline = Pipeline([
    ('bow', CountVectorizer()),  # strings to token integer counts
   # ('tfidf', TfidfTransformer(use_idf=True, smooth_idf=True)),  # integer counts to weighted TF-IDF scores
    ('classifier', MultinomialNB()),  # train on TF-IDF vectors w/ Naive Bayes classifier
])

In [11]:
#Fit the training data
pipeline.fit(train_df['CONTENT'],train_df['CLASS'])

Pipeline(steps=[('bow', CountVectorizer()), ('classifier', MultinomialNB())])

#### Use that filter to check the accuracy of the remaining dataset.

In [12]:
#Predict on the testing data
predictions = pipeline.predict(test_df['CONTENT'])

#Print the classification report and accuracy score
print(classification_report(predictions,test_df['CLASS']))
print("accuracy:", round(accuracy_score(predictions,test_df['CLASS']), 4))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88       186
           1       0.90      0.85      0.87       184

    accuracy                           0.88       370
   macro avg       0.88      0.88      0.88       370
weighted avg       0.88      0.88      0.88       370

accuracy: 0.8757


In [13]:
from sklearn.feature_extraction.text import CountVectorizer
# Multiple documents
tweet = ["Next I'm buying Coca-Cola to put the cocaine back in"] 
# create the transform
vectorizer = CountVectorizer()
# tokenize and build vocab
vectorizer.fit(tweet)
# summarize
print(sorted(vectorizer.vocabulary_))

['back', 'buying', 'coca', 'cocaine', 'cola', 'in', 'next', 'put', 'the', 'to']


In [14]:
# encode document
vector = vectorizer.transform(tweet)
# summarize encoded vector
print(vector.shape)
print(vector.toarray())

(1, 10)
[[1 1 1 1 1 1 1 1 1 1]]
